In [5]:
import torch
import torch.nn as nn
from gensim.models import KeyedVectors

## 词嵌入层

In [6]:
embedding = nn.Embedding(
    num_embeddings=100000,
    embedding_dim=300,
)

In [7]:
print(embedding.weight.shape)

torch.Size([100000, 300])


## 词向量应用：加载到词嵌入层

In [8]:
wv_model = KeyedVectors.load_word2vec_format("data/word2vec.model")
# 1.加载预训练好的词向量

In [9]:
# 2.基于预训练的词向量矩阵，创建Embedding层
num_embeddings = len(wv_model.key_to_index)
embedding_dim = wv_model.vector_size
print(embedding_dim)
print(num_embeddings)

100
34576


In [10]:
# 3。 向前传播测试
id = wv_model.key_to_index['地铁']
input = torch.tensor([id])
# 前向传播，得到词向量
vector = embedding(input)
print(vector)
print(vector.shape)

tensor([[ 0.0171, -0.0332, -1.3975,  0.8573, -1.7873,  1.2303, -0.2849, -0.8731,
          0.5760, -0.4590,  0.7226, -0.5487, -0.3502, -1.0756, -0.6228,  1.4760,
          1.0001,  0.7240,  1.5197,  0.7543,  0.7344, -0.8770, -0.1921,  0.5030,
          0.0634,  0.6029, -1.0668,  0.7782,  0.2169, -0.6695, -0.1280, -1.3572,
          0.1136,  0.2480,  1.4513,  1.1775, -1.1456, -1.5929, -0.5760, -1.6370,
          0.0967,  0.1103, -0.4480, -0.3554, -1.0474,  0.7209, -0.2217,  1.9382,
         -1.5783,  1.3910,  0.0696, -0.1883,  0.7904, -1.0736, -0.4484,  0.9038,
          1.2149, -1.1178, -1.3632, -0.4436,  1.8363,  1.5549,  0.0483,  0.5891,
         -0.8740, -0.3193, -0.4056, -2.0290, -0.6560, -1.3706,  1.9038,  0.1769,
          0.1518,  0.1229,  0.2817, -1.9271, -0.2992,  0.2613, -1.0503,  0.5528,
         -0.2317, -0.7775, -0.5870,  0.9201, -2.7852,  1.2172, -1.0940,  0.8184,
          1.2294,  0.8280, -0.6744,  0.5109, -0.5303, -1.2833,  0.7344,  1.0592,
         -0.0084,  1.7832,  

## 综合应用实例——直接使用训练好的Win得到词向量

In [1]:
import jieba

In [11]:
text = "我喜欢乘坐地铁"

In [12]:
# 1.分词
tokens = jieba.lcut(text)
print(tokens)

['我', '喜欢', '乘坐', '地铁']


In [13]:
# 2.编码，将token转化为id
word2id = wv_model.key_to_index
ids = [ word2id[token] for token in tokens ]
print(ids)

[7, 49, 5819, 1519]


In [14]:
# 3.转化为张量，作为模型输入
input = torch.tensor(ids)
print(input.shape)

torch.Size([4])


In [15]:
# 4,向前传播，输入id列表，得到词向量列表
vector = embedding(input)
print(vector.shape)

torch.Size([4, 300])


In [16]:
print(vector)

tensor([[-0.1380,  0.3245, -0.4274,  ...,  0.8542, -0.4733, -0.0588],
        [-0.2860, -0.0717, -1.4804,  ...,  0.3120, -0.0571, -1.5831],
        [-1.0476,  1.0385,  0.6394,  ...,  1.4808, -0.2318,  1.5603],
        [ 0.0171, -0.0332, -1.3975,  ...,  0.0105, -1.1491,  0.8917]],
       grad_fn=<EmbeddingBackward0>)


## OOV问题

In [25]:
text = ("我喜欢乘坐宇宙飞船")

In [26]:
# 新增特殊token
unk_token = '<UNK>'
id2word = [unk_token]+ wv_model.index_to_key
word2id = { word:id for id, word in enumerate(id2word) }

In [27]:
print(id2word)
print(len(word2id))

['<UNK>', '，', '的', '。', '了', '！', '很', '是', '我', '也', '好', '不', ',', '都', '就', '买', '不错', '还', '在', '有', '没有', '酒店', '用', '.', '京东', '和', '说', '房间', '给', '可以', '、', '这', '就是', '到', '非常', '一个', '感觉', '还是', '？', '没', '这个', '服务', '质量', '人', '比较', '上', '苹果', '要', '看', '去', '喜欢', '东西', '太', '不是', '又', '小', '但', '我们', '大', '价格', '让', '什么', '但是', '吧', '你', '差', '住', '而且', '多', '个', '…', '知道', '自己', '再', '才', '满意', '会', '啊', '有点', '：', '对', '比', '不好', '挺', '问题', '快递', '收到', '能', '吃', '手机', '2', '裤子', '来', '时候', '一般', '快', '还有', '一样', '?', '入住', '方便', '后', '真的', '这样', '一直', '~', '包装', '；', '蒙牛', '!', '物流', '着', '速度', '过', '以后', '不过', '3', '舒服', '客服', '特别', '一次', '不能', '穿', '*', '很多', '他', '味道', '送', '1', '）', '现在', '一点', '因为', '前台', '购买', '下次', '屏幕', '觉得', '已经', '早餐', '差评', '（', '想', '本书', '中', '时', '不会', '垃圾', '月', '第一次', '里', '跟', '把', '这次', '这么', '值得', '支持', '如果', '被', '效果', '呢', '得', '4', '便宜', '应该', '“', '很快', '”', '最', '起来', '发现', '一', '点', '等', '所以', '高', '希望', '大家', '设施', '失望', '怎么', '

In [28]:
# 基于预训练的词向量矩阵，创建embedding层
embedding_matrix = torch.cat([torch.zeros(1,embedding_dim), torch.tensor(wv_model.vectors)])
embedding = nn.Embedding.from_pretrained(
    embeddings= embedding_matrix,
    freeze=False,
)

In [29]:
print(embedding.weight.shape)

torch.Size([34577, 100])


In [30]:
# 1.分词
tokens = jieba.lcut(text)
print(tokens)
# 2.编码，将token转化为id
ids = [word2id.get(token,word2id[unk_token])for token in tokens]
print(ids)

['我', '喜欢', '乘坐', '宇宙飞船']
[8, 50, 5820, 0]


In [31]:
# 3.转化为张量，作为模型输入
input = torch.tensor(ids)
print(input.shape)
# 4,向前传播，输入id列表，得到词向量列表
vector = embedding(input)
print(vector.shape)
print(vector)

torch.Size([4])
torch.Size([4, 100])
tensor([[ 0.2112, -0.1590,  0.0973, -0.0493, -0.1683, -0.7221,  0.3419, -0.0379,
         -0.1448, -0.0612,  0.3366, -0.0867,  0.0035, -0.5951, -0.0631, -0.3207,
          0.1631, -0.3270, -0.2429, -0.3848,  0.5536, -0.0455,  0.1634, -0.0825,
          0.0050,  0.1737,  0.0384,  0.0559, -0.1042, -0.1958,  0.3296,  0.1493,
          0.1083, -0.0026, -0.1618,  0.4093,  0.2973, -0.1736, -0.3101, -0.2670,
          0.0689, -0.2147,  0.1387,  0.0041,  0.3470,  0.0525, -0.0724,  0.2041,
          0.1716,  0.2824, -0.3076, -0.0129,  0.2700, -0.1376, -0.2677,  0.5080,
          0.3297, -0.0367, -0.1981,  0.1392,  0.4486, -0.3152, -0.0409, -0.4297,
         -0.1107,  0.3152,  0.0666,  0.2846, -0.2860, -0.0504, -0.3343, -0.2099,
          0.3759, -0.1011,  0.2830,  0.4195,  0.2238, -0.2653,  0.0268, -0.1461,
         -0.4294,  0.2045,  0.1784,  0.1234, -0.3461, -0.4357,  0.0240,  0.1657,
          0.5142, -0.3246,  0.2324, -0.1809, -0.5829,  0.0781,  0.5251, 